# Publish a Model: Serving in Python


- Author: Jaya Srinivas
- Date: 2026-08
- Dataset: Seaborn Iris
- Target: species
  
Run all cells top to bottom (**Run All**) before pushing to GitHub.

## M6. Serving

This notebook trains a model, saves it, reloads it, and 
tests the **serving core** - the function a web server would call - in-process. 

It does **not** launch a blocking server inside the notebook; 
the real server is shown for you to run with `uv`. 

The analyst decides the API contract and how to handle bad input.

Modules 3-5 built and evaluated models.

This module **publishes** one: 

- a saved artifact and 
- a testable prediction function that a server wraps.

## Overview

This project uses the iris dataset.
We choose to predict the target `species`.
This target is a **discrete category**, so we have a:

- supervised ML problem (because we've chosen a target)
- a classification problem (because our target is a category)

The features are four flower measurements (in cm):
`sepal_length`, `sepal_width`, `petal_length`, `petal_width`.
The three possible species are `setosa`, `versicolor`, and `virginica`.

## A. Prepare the Project Environment (.venv/)

- Open one project in VS Code at a time.
- Prepare the .venv/: specify Python version and install / upgrade dependencies listed in `pyproject.toml`.
- Open an integrated terminal (PowerShell if Windows) in the **root project** folder and run:

```shell
uv self update
uv python pin 3.14
uv lock --upgrade
uv sync --extra dev --extra docs --upgrade
```


## B. Select the Notebook Kernel

- Click on the **Select Kernel** name in the top-right corner of the notebook interface.
- Choose Python Environments... /
- Choose the recommended local .venv/ from the drop-down menu.
- This will create a new kernel for the notebook and allow the notebook to use packages installed in the .venv/ environment.

## C. Working in Notebooks (Custom Notes)

- To run a cell, press **Ctrl+Enter** (or **Cmd+Enter** on Mac) when done editing the cell.
- Change the type of a cell (e.g., code or markdown) by looking in the lower left corner of the notebook interface.
- Rearrange cells by dragging and dropping them within the notebook.

See [Run Jupyter Notebooks](https://denisecase.github.io/pro-analytics-02/workflow-b-apply-example-project/run-notebook/) for:

- how to **copy a notebook**
- how to release a `project.log` file
- how to deal with a **stuck kernel**
- etc.

## Section 0. Publishing a Model

A trained model is useful if something can call it on new data. 

Publishing it means three things:

1. **Persist** the trained model to a file (here, with `joblib`) so you do not retrain
   on every request, and reload it the same way.
2. **A serving core** - a small, pure function that takes inputs, validates them,
   calls the model, and returns a result. Keeping this logic in one tested function
   (separate from the web framework) is what makes it reliable and easy to test.
3. **A server** - a thin web layer (here, FastAPI) that receives a request, calls the
   serving core, and returns the result as JSON. The notebook tests the core; the
   server is run separately with `uv`.

The judgment is the **contract**: 

what inputs are required, 
what happens on missing or malformed input, and 
what shape the response takes. 

A server that crashes on bad input is fragile.

## Section 1. Project Setup and Imports

In [ ]:
# Section 1a. DECLARE IMPORTS

from importlib.metadata import version  # to verify
import logging  # for type hinting
import platform  # to verify
from typing import Any  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import joblib
import pandas as pd

from mlstudio.model_builder_custom import (
    DATASET_NAME,
    MODEL_PATH,
    TARGET_COL,
    load_data,
    save_model,
    split_data,
    summarize,
    train_model,
)
from mlstudio.serve_custom import predict_from_features

# Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK

LOG: logging.Logger = get_logger("M06", level="DEBUG")
log_header(LOG, "M06")


#  Section 1c. USE THE LOGGER TO VERIFY IMPORTS

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")

# Section 1d. SET PANDAS DISPLAY CONFIGURATION

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Section 1e. GLOBAL CONSTANTS AND CONFIGURATION

# CUSTOM: where the published model artifact is written.
LOG.info(f"Model artifact will be saved to: {MODEL_PATH}")

## Section 2. Load and Prepare the Model

In [ ]:
# Section 2. Load the Data

LOG.info(f"Loading dataset: {DATASET_NAME}")
df_model: pd.DataFrame = load_data()
LOG.info(f"Model rows: {df_model.shape[0]}")
LOG.info(f"Classes in '{TARGET_COL}': {sorted(df_model[TARGET_COL].unique())}")

2026-08-07 19:11:40 | INFO | M06 | Loading dataset: penguins
2026-08-07 19:11:40 | INFO | M06 | Loading dataset: penguins
2026-08-07 19:11:40 | INFO | M06 | Loaded: 344 rows, 7 columns
2026-08-07 19:11:40 | INFO | M06 | Model rows (after dropping missing): 342
2026-08-07 19:11:40 | INFO | M06 | Model rows: 342
2026-08-07 19:11:40 | INFO | M06 | Classes in 'species': ['Adelie', 'Chinstrap', 'Gentoo']


## Section 3. Split into Train and Test

In [ ]:
# Section 3. Split into Train and Test

X_train, X_test, y_train, y_test = split_data(df_model)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances:  {len(X_test)}")

2026-08-07 19:11:40 | INFO | M06 | Train instances: 273
2026-08-07 19:11:40 | INFO | M06 | Test instances:  69
2026-08-07 19:11:40 | INFO | M06 | Train instances: 273
2026-08-07 19:11:40 | INFO | M06 | Test instances:  69


## Section 4. Train, Save, Reload Model 

In [ ]:
# Section 4. Train, Save, and Reload

model = train_model(X_train, y_train)
save_model(model)
model = joblib.load(MODEL_PATH)
LOG.info(f"Reloaded model from: {MODEL_PATH}")

2026-08-07 19:11:40 | INFO | M06 | Training RandomForestClassifier on 273 instances
2026-08-07 19:11:41 | INFO | M06 | Training complete
2026-08-07 19:11:41 | INFO | M06 | Saved model to: artifacts/model.joblib
2026-08-07 19:11:41 | INFO | M06 | Reloaded model from: artifacts/model.joblib


## Section 5. Test the Serving Core

In [ ]:
#  Section 5. Test the Serving Core

# valid payload - should return a prediction
good_payload: dict[str, Any] = {
    "sepal_length": 6.7,
    "sepal_width": 3.1,
    "petal_length": 4.7,
    "petal_width": 1.5,
}

# show confidence alongside the label
proba = model.predict_proba(
    [
        [
            good_payload["sepal_length"],
            good_payload["sepal_width"],
            good_payload["petal_length"],
            good_payload["petal_width"],
        ]
    ]
)[0]
proba_by_class = dict(zip(model.classes_, proba.round(3), strict=True))
LOG.info(f"Class probabilities -> {proba_by_class}")


result: dict[str, Any] = predict_from_features(model, good_payload)
LOG.info(f"Valid payload -> {result}")

# invalid payload - should raise a clean ValueError, not crash
bad_payload: dict[str, Any] = {"sepal_length": 6.7}
try:
    predict_from_features(model, bad_payload)
    LOG.warning("Expected a ValueError for the bad payload but none was raised.")
except ValueError as exc:
    LOG.info(f"Invalid payload handled cleanly -> ValueError: {exc}")

## Section 6. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [ ]:
#  Summary

# Python summary
summarize()

2026-08-07 19:11:41 | INFO | M06 | ========================
2026-08-07 19:11:41 | INFO | M06 | SUMMARY
2026-08-07 19:11:41 | INFO | M06 | ========================
2026-08-07 19:11:41 | INFO | M06 | Dataset:  penguins
2026-08-07 19:11:41 | INFO | M06 | Target:   species
2026-08-07 19:11:41 | INFO | M06 | Features: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
2026-08-07 19:11:41 | INFO | M06 | Artifact: artifacts/model.joblib
2026-08-07 19:11:41 | INFO | M06 | ========================
